 # Ekstrakcija simptoma iz korisničkog unosa (NLP)
Notebook demonstrira ekstrakciju simbola iz korisničkog unosa uz pomoć LLM-a.

In [ ]:
%pip install groq python-dotenv

## Konfiguracija

Koristi se LLM model **meta-llama/llama-4-scout-17b-16e-instruct**, razvijen od strane kompanije Meta. Karakteristike modela:

1. Pripada grupi instruction-tuned modela, optimizovanih za praćenje uputstava i strukturisanih odgovora,

2. Srednji model po veličini sa 17 milijardi parametara, treniran sa 16 slojeva optimizovanih za zadate instrukcije,

3. Pogodan je za struktuiranu ekstrakciju informacija, generisanje JSON formata i praćenje striktnih pravila.

Rezultati testiranja kao i razlog zašto se koristi gore pomenuti model su objašnjeni u **NLP-test.docx**.

In [2]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("LLM_API_KEY")

client = Groq(api_key=groq_api_key)

MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

## Sistemski prompt

- **Role**: definiše identitet modela kao Medical Informatics Specialist, sa fokusom na striktno strukturirano mapiranje simptoma na ontologiju (SYMP/DOID kompatibilno). Model se ponaša kao ekspert iz oblasti medicine.

- **Objective**: krajnji cilj prompta je konverzija slobodnog teksta pacijenta na srpskom jeziku u strukturisanu listu atomičnih, kanoničkih medicinskih simptoma na engleskom jeziku.

- **Critical Normalization Rule / Canonical Normalization**: Svaki simptom se mapira na jedan standardni kanonički termin.Sinonimi se spajaju u jedan preferirani termin. Primeri daju uvid u preferirane kanoničke oblike (npr. runny nose → rhinorrhea).

- **Task**: Analiza srpskog ulaznog teksta, ekstrakcija samo eksplicitno navedenih pozitivnih simptoma, atomizacija složenih opisa simptoma i normalizacija u kanoničke termine.

- **General rules**: Ne uključivati dijagnoze, uzroke, trajanje, ili zaključke. Ne dodavati medicinski povezane koncepte koji nisu eksplicitno navedeni.

- **Negation**: Kontrola negacije je striktna: simptomi koje pacijent negira potpuno se izbacuju iz rezultata.

- **Granularity / Atomic decomposition**: Sprečava spajanje više simptoma u jednu frazu. Zadržava klinički relevantne modifikatore samo ako menjaju značenje (sharp chest pain vs chest pain). Ignoriše reči koje samo ukazuju na intenzitet (severe, high, mild).

- **Anatomical localization**: Omogućava precizno mapiranje simptoma na anatomske lokacije. Sprečava generičke termine poput bol bez preciziranja lokacije u kom predelu je bol.

- **Output Requirements**: Izlaz je isključivo validan JSON, u formatu: {"symptoms": ["symptom_1", "symptom_2"]} Bez komentara, objašnjenja ili dodatnog teksta. Ako nema simptoma ili su svi negirani: {"symptoms": []}

In [4]:
from pathlib import Path

BASE_DIR = Path.cwd()
prompt_path = BASE_DIR / "tests" / "nlp-extraction-test" / "prompts" / "symptom-extraction-prompt-3"

SYSTEM_PROMPT = prompt_path.read_text(encoding="utf-8")

## Funkcija za ekstrakciju simptoma

Pozivanje Groq API-ja:

- Funkcija koristi client.chat.completions.create metod iz Groq SDK da pošalje ulaz korisnika LLM-u,

- U porukama se šalju dve uloge:

    **system** – sadrži SYSTEM_PROMPT koji definiše pravila ekstrakcije i normalizacije simptoma.

    **user** – sadrži slobodan tekst pacijenta.

- Parametri modela:

    **temperature=0.0** → deterministički izlaz, bez nasumičnih varijacija.

    **top_p=1** → puna distribucija verovatnoće za predikciju.

    **max_tokens=800** → maksimalni broj tokena u odgovoru.

    **stream=False** → kompletan odgovor odjednom.

- Parsiranje rezultata:

    Odgovor modela se očekuje u JSON formatu, prema pravilima SYSTEM_PROMPT.

    Funkcija koristi json.loads da parsira JSON i izvuče listu simptoma.

- Vraćanje rezultata:

    Ako JSON sadrži ključ "symptoms", funkcija vraća njegovu vrednost kao listu stringova.

    Ako dođe do greške ili izlaz nije u očekivanom formatu, funkcija hvata Exception i vraća praznu listu.

In [5]:
import json
from typing import List

def extract_symptoms_groq(user_input: str) -> List[str]:
    try:
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ],
            temperature=0.0,
            top_p=1,
            max_tokens=800,
            stream=False,
        )

        result = json.loads(completion.choices[0].message.content)
        return result.get("symptoms", [])

    except Exception as e:
        print(f"Error {e}")
        return []

## Test podaci

Učitavanje test podataka.

In [6]:
test_cases_path = BASE_DIR / "tests" / "nlp-extraction-test" / "symptom-extraction-test-data.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

## Evaluacija rezultata

Glavna metoda koja radi evaluaciju dobijenih rezultata je metoda **evaluate_case**. 

In [7]:
def normalize(text: str) -> str:
    if not text:
        return ""
    return text.lower().strip()


def evaluate_case(extracted, expected, not_expected=None):
    extracted_norm = set(normalize(x) for x in extracted) if extracted else set()
    expected_norm = set(normalize(x) for x in expected) if expected else set()

    if len(extracted_norm) == 0 and len(expected_norm) == 0:
        return {
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0,
            "false_positive": [],
            "false_negative": [],
            "negation_error": False
        }

    true_positive = extracted_norm & expected_norm
    false_positive = extracted_norm - expected_norm
    false_negative = expected_norm - extracted_norm

    precision = len(true_positive) / len(extracted_norm) if extracted_norm else 0.0
    recall    = len(true_positive) / len(expected_norm) if expected_norm else 0.0
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    negation_error = False
    if not_expected:
        not_expected_norm = {normalize(item) for item in not_expected}
        if not_expected_norm & extracted_norm:
            negation_error = True

    return {
        "precision": round(precision, 3),
        "recall":    round(recall, 3),
        "f1":        round(f1, 3),
        "false_positive": sorted(list(false_positive)),
        "false_negative": sorted(list(false_negative)),
        "negation_error": negation_error
    }

## Ulazna tačka za testiranje

Prilikom svakog poziva pravi se pauza od 10 sekundi zbog **ograničenja broja poziva koja može da se uputi ka Groq API**! Ukoliko neki poziv ne uspe, pokušaće se još pet puta (retry mehanizam), a sve u cilju da se izbegne slučaj u kojem poziv nije uspešan a mi takav odgovor uzimamo u razmatranje. U tom slučaju ne bismo imali dobre rezultate.

In [10]:
import time
import statistics

def test_symptom_extraction(max_retries=5, sleep_base=2):
    total_precision = []
    total_recall = []
    total_f1 = []
    failed_cases = []

    for i, case in enumerate(test_cases):
        extracted = None
        retry_count = 0
        while retry_count < max_retries and extracted is None:
            try:
                extracted = extract_symptoms_groq(case["input"])
                break
            except Exception as e:
                err_str = str(e)
                print(f"Error on {case['id']}: {err_str}")
                if "429" in err_str or "rate limit" in err_str.lower():
                    retry_count += 1
                    wait_time = sleep_base ** retry_count
                    print(f"Rate limit (429) → retry {retry_count}/{max_retries} after {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print("Non-rate-limit error → skipping to empty list")
                    extracted = []
                    break

        if extracted is None:
            print(f"Failed after {max_retries} retries for {case['id']}")
            extracted = []
            failed_cases.append(case['id'])

        result = evaluate_case(
            extracted,
            case["expected"],
            case.get("not_expected")
        )

        total_precision.append(result["precision"])
        total_recall.append(result["recall"])
        total_f1.append(result["f1"])

        print(f"\n--- {case['id']} ---")
        print("Extracted (normalized):", sorted(extracted))
        print("Expected:             ", sorted(case["expected"]))
        print(f"Precision: {result['precision']:.3f}")
        print(f"Recall:    {result['recall']:.3f}")
        print(f"F1:        {result['f1']:.3f}")
        print("False Positive:", result["false_positive"])
        print("False Negative:", result["false_negative"])
        print("Negation Error:", result["negation_error"])

        time.sleep(10)

    if failed_cases:
        print(f"\nFailed cases: {failed_cases}")

    avg_precision = statistics.mean(total_precision) if total_precision else 0
    avg_recall    = statistics.mean(total_recall)    if total_recall    else 0
    avg_f1        = statistics.mean(total_f1)        if total_f1        else 0

    print("\n====================")
    print(f"Average Precision: {avg_precision:.3f}")
    print(f"Average Recall:    {avg_recall:.3f}")
    print(f"Average F1-score:  {avg_f1:.3f}")

In [11]:
test_symptom_extraction()


--- CANONICAL_001 ---
Extracted (normalized): ['fever', 'headache']
Expected:              ['fever', 'headache']
Precision: 1.000
Recall:    1.000
F1:        1.000
False Positive: []
False Negative: []
Negation Error: False

--- CANONICAL_002 ---
Extracted (normalized): ['cough', 'fever', 'headache']
Expected:              ['cough', 'fever', 'headache']
Precision: 1.000
Recall:    1.000
F1:        1.000
False Positive: []
False Negative: []
Negation Error: False

--- CANONICAL_003 ---
Extracted (normalized): ['diarrhea', 'nausea', 'vomiting']
Expected:              ['diarrhea', 'nausea', 'vomiting']
Precision: 1.000
Recall:    1.000
F1:        1.000
False Positive: []
False Negative: []
Negation Error: False

--- CANONICAL_004 ---
Extracted (normalized): ['fever', 'joint pain', 'rash']
Expected:              ['fever', 'joint pain', 'rash']
Precision: 1.000
Recall:    1.000
F1:        1.000
False Positive: []
False Negative: []
Negation Error: False

--- CANONICAL_005 ---
Extracted (no

KeyboardInterrupt: 